# 🚐 CareMax Pickup Timing Analysis
### ⏱ Time Picked Up Before Appointment (Histogram + Pareto View)

This notebook computes how early patients are picked up before scheduled appointments, groups results into 10-minute intervals (0–210 minutes), flags >60-minute early pickups, and visualizes the distribution using a Pareto-style chart in R.

## 1️⃣ Load Required R Libraries
We load `ggplot2` for plotting and `scales` for percentage formatting on the secondary axis.

In [ ]:
library(ggplot2)
library(scales)

## 2️⃣ (Optional) Sample Data Illustration
This small sample shows how NA values are handled in a simple subtraction example. It is not used in the final plot, but kept here for reference/testing.

In [ ]:
# Sample data (assuming values are in seconds)
data <- data.frame(
  column1 = c(5, NA, 8, 2),   # seconds, with NA
  column2 = c(2, 4, NA, 1)    # seconds, with NA
)

head(data)

## 3️⃣ Load Trip Data (`all_data1`)
Load your main dataset, which should contain at least the columns `Appointment` and `Req.Pickup` in seconds.

> 🔧 Make sure `all_data1.csv` is in the same folder as this notebook, or adjust the path accordingly.

In [ ]:
all_data1 <- read.csv("all_data1.csv")
head(all_data1)

## 4️⃣ Compute Minutes Picked Up Before Appointment
Here we subtract the required pickup time from the appointment time, convert seconds to minutes, and flag whether the pickup occurred more than 60 minutes before the appointment.

In [ ]:
# Subtract values from Appointment and Req.Pickup (convert to minutes)
subtracted_minutes <- (all_data1$Appointment - all_data1$Req.Pickup) / 60  # Convert to minutes, NAs will be preserved

# Create a data frame to store information for Pareto chart
pareto_data <- data.frame(
  Minutes = subtracted_minutes,
  Greater_than_60 = ifelse(subtracted_minutes > 60, "Yes", "No")
)

# Convert Minutes column to numeric
pareto_data$Minutes <- as.numeric(pareto_data$Minutes)

# Filter out N/A values
pareto_data <- pareto_data[!is.na(pareto_data$Minutes), ]

head(pareto_data)

## 5️⃣ Create 10-Minute Intervals for the X-Axis
We define breaks from 0 to 210 minutes in 10-minute steps and label each interval (e.g., `0-10`, `10-20`, …).

In [ ]:
# Create 10-minute intervals for x-axis
interval_breaks <- seq(0, 210, by = 10)

# Cut data into 10-minute intervals
pareto_data$Interval <- cut(
  pareto_data$Minutes,
  breaks = interval_breaks,
  labels = paste(interval_breaks[-length(interval_breaks)], "-", interval_breaks[-1])
)

head(pareto_data$Interval)

## 6️⃣ Count Frequencies per Interval and 60+ Flag
We now compute how many trips fall into each 10-minute interval, split by whether they exceed the 60-minute early-pickup threshold (`Yes`/`No`).

In [ ]:
# Calculate counts
counts <- table(pareto_data$Interval, pareto_data$Greater_than_60)

# Convert to data frame
pareto_counts <- as.data.frame(counts)
head(pareto_counts)

## 7️⃣ Compute Cumulative Counts for Pareto Line
Here we add a `cumulative` column representing the running total per interval, used as the Pareto line over the bars.

In [ ]:
# Calculate cumulative percentages (cumulative counts)
pareto_counts$cumulative <- ave(pareto_counts$Freq, pareto_counts$Var1, FUN = cumsum)

head(pareto_counts)

## 8️⃣ Build Pareto Chart — Histogram + Cumulative Line
Finally, we create the Pareto-style chart with trip frequency per interval and cumulative trend on a secondary axis.

In [ ]:
# Create Pareto chart
pareto_plot <- ggplot(pareto_counts, aes(x = Var1, y = Freq, fill = Var2)) +
  geom_bar(stat = "identity") +
  geom_text(aes(label = Freq), vjust = -0.5, size = 3) +
  geom_line(aes(x = Var1, y = cumulative, group = Var2, color = Var2), linetype = "dashed") +
  geom_point(aes(x = Var1, y = cumulative, group = Var2, color = Var2), size = 2) +
  scale_y_continuous(sec.axis = sec_axis(~./sum(pareto_counts$Freq), labels = scales::percent_format())) +
  labs(
    x = "Time Interval (minutes)",
    y = "Frequency",
    fill = "Greater than 60 minutes?",
    color = "Greater than 60 minutes?",
    title = "Time before appointment in which pickup was performed"
  ) +
  theme_minimal() +
  theme(legend.position = "bottom")

# Print Pareto chart
print(pareto_plot)